# Vortex D4 — Fairness Audit & Mitigation (MovieLens 1M, Spark ML)

**Extension of the D3 logistic-regression rating classifier into a demographic
fairness study**, written up for an IEEE conference (OMLET 2026).

This notebook adds, on top of the D3 pipeline:

1. **Leakage-safe retrain** — user/movie averages computed on the training split only.
2. **Group fairness metrics** across **Gender** and **Age**: Demographic Parity Difference
   (DPD), Disparate Impact (DI), Equal-Opportunity Difference (EOD).
3. **Three mitigations**, all post-processing on the same model:
   - sensitive-feature removal ("fairness through unawareness"),
   - single-attribute per-group thresholding (Gender *or* Age),
   - **intersectional (multi-attribute) thresholding** that equalizes selection rate
     across Gender×Age cells simultaneously *(new method contribution)*.
4. **Cross-seed significance testing** over 8 random splits — mean±std and paired
   significance tests, so the headline gaps are not single-seed artifacts.

> **Reproduce:** Java 17 + a Python 3.11 venv with `pyspark==3.5.3 scipy pandas matplotlib`.
> If a Homebrew Spark is installed, **unset `SPARK_HOME`** before launching Jupyter or the
> bundled Spark jars will clash.

## 0. Environment & Spark session

In [1]:
import os, sys, json, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault(
    "JAVA_HOME", "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
)
# Guard against a Homebrew Spark clashing with the venv's bundled jars.
for v in ("SPARK_HOME", "PYSPARK_SUBMIT_ARGS"):
    os.environ.pop(v, None)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array

spark = (
    SparkSession.builder.appName("MovieLens Fairness D4")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
DATA = "./ml-1m"
print("Spark", spark.version)

26/07/20 21:42:57 WARN Utils: Your hostname, Mostafizurs-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.171 instead (on interface en0)
26/07/20 21:42:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/20 21:42:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3


## 1. Load, clean, and engineer label-independent features

We load the three `::`-delimited files with explicit schemas, drop duplicate
`(UserID, MovieID)` pairs, keep valid ratings, and enforce referential integrity.
The binary label is `high_rating = (Rating >= 4)`. Movie-level features that do **not**
depend on the label (`release_year`, `num_genres`, `movie_age`) are computed here; the
label-dependent averages are computed later, **after** the split, to avoid leakage.

In [2]:
def load():
    users_schema = StructType([
        StructField("UserID", IntegerType()), StructField("Gender", StringType()),
        StructField("Age", IntegerType()), StructField("Occupation", IntegerType()),
        StructField("Zip", StringType()),
    ])
    ratings_schema = StructType([
        StructField("UserID", IntegerType()), StructField("MovieID", IntegerType()),
        StructField("Rating", IntegerType()), StructField("Timestamp", IntegerType()),
    ])
    movies_schema = StructType([
        StructField("MovieID", IntegerType()), StructField("Title", StringType()),
        StructField("Genres", StringType()),
    ])
    opt = dict(sep="::", header="false", quote='"', escape='"', multiLine="true")
    users = spark.read.options(**opt).schema(users_schema).csv(f"{DATA}/users.dat")
    ratings = spark.read.options(**opt).schema(ratings_schema).csv(f"{DATA}/ratings.dat")
    movies = spark.read.options(**opt).schema(movies_schema).csv(f"{DATA}/movies.dat")
    return users, ratings, movies

users, ratings, movies = load()

ratings = (ratings.dropDuplicates(["UserID", "MovieID"])
           .filter((F.col("Rating") >= 1) & (F.col("Rating") <= 5))
           .join(users.select("UserID"), "UserID", "inner")
           .join(movies.select("MovieID"), "MovieID", "inner"))

movies = (movies
          .withColumn("release_year",
                      F.regexp_extract("Title", r"\((\d{4})\)", 1).cast(IntegerType()))
          .withColumn("num_genres", F.size(F.split("Genres", r"\|")))
          .withColumn("movie_age", F.lit(2000) - F.col("release_year")))

df = (ratings
      .join(users, "UserID")
      .join(movies.select("MovieID", "release_year", "num_genres", "movie_age"), "MovieID")
      .withColumn("high_rating", (F.col("Rating") >= 4).cast(IntegerType()))
      .withColumn("gender_encoded", (F.col("Gender") == "F").cast(IntegerType()))
      .cache())

n_total = df.count()
print(f"Ratings after cleaning: {n_total:,}")
print(f"Users: {users.count():,}   Movies: {movies.count():,}")
print(f"Positive-class (high_rating) prevalence: {df.select(F.avg('high_rating')).first()[0]:.3f}")

Ratings after cleaning: 1,000,209
Users: 6,040   Movies: 3,883


Positive-class (high_rating) prevalence: 0.575


## 2. Pipeline helpers

`add_train_derived_features` computes user/movie mean-rating and count features on the
**training split only** and maps them onto both splits (unseen users/movies in test fall
back to the global training mean) — this is the leakage fix. `train_lr` is the D3
`VectorAssembler → StandardScaler → LogisticRegression` pipeline. `group_fairness`
returns per-group selection rate, TPR, FPR, accuracy and the DPD/DI/EOD summary.

In [3]:
BASE_FEATURES = [
    "Age", "Occupation", "user_avg_rating", "movie_avg_rating",
    "movie_popularity", "user_rating_count", "gender_encoded",
    "num_genres", "release_year", "movie_age",
]

def add_train_derived_features(train_df, test_df):
    global_mean = train_df.select(F.avg("Rating")).first()[0]
    user_avg = train_df.groupBy("UserID").agg(
        F.avg("Rating").alias("user_avg_rating"),
        F.count("*").alias("user_rating_count"))
    movie_avg = train_df.groupBy("MovieID").agg(
        F.avg("Rating").alias("movie_avg_rating"),
        F.count("*").alias("movie_popularity"))
    def attach(d):
        d = d.join(user_avg, "UserID", "left").join(movie_avg, "MovieID", "left")
        return d.fillna({"user_avg_rating": global_mean, "movie_avg_rating": global_mean,
                         "user_rating_count": 0, "movie_popularity": 0})
    return attach(train_df), attach(test_df), global_mean

def add_leaky_features(full_df):
    ua = full_df.groupBy("UserID").agg(F.avg("Rating").alias("user_avg_rating"),
                                       F.count("*").alias("user_rating_count"))
    mv = full_df.groupBy("MovieID").agg(F.avg("Rating").alias("movie_avg_rating"),
                                        F.count("*").alias("movie_popularity"))
    return full_df.join(ua, "UserID").join(mv, "MovieID")

def train_lr(train_df, feature_cols):
    asm = VectorAssembler(inputCols=feature_cols, outputCol="raw_features",
                          handleInvalid="skip")
    sc = StandardScaler(inputCol="raw_features", outputCol="features")
    lr = LogisticRegression(featuresCol="features", labelCol="high_rating", maxIter=100)
    return Pipeline(stages=[asm, sc, lr]).fit(train_df)

def score(model, test_df, threshold=0.5):
    return (model.transform(test_df)
            .withColumn("p1", vector_to_array("probability")[1])
            .withColumn("pred", (F.col("p1") >= threshold).cast("double")))

def overall_metrics(pred):
    tp = pred.filter("high_rating=1 AND pred=1.0").count()
    fp = pred.filter("high_rating=0 AND pred=1.0").count()
    tn = pred.filter("high_rating=0 AND pred=0.0").count()
    fn = pred.filter("high_rating=1 AND pred=0.0").count()
    n = tp + fp + tn + fn
    acc = (tp + tn) / n if n else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, n=n)

def group_fairness(pred, group_col):
    rows = (pred.groupBy(group_col).agg(
        F.count("*").alias("n"),
        F.avg("pred").alias("selection_rate"),
        F.avg(F.when(F.col("high_rating") == 1, F.col("pred"))).alias("tpr"),
        F.avg(F.when(F.col("high_rating") == 0, F.col("pred"))).alias("fpr"),
        F.avg((F.col("pred") == F.col("high_rating")).cast("double")).alias("accuracy"),
    ).orderBy(group_col).collect())
    groups = {str(r[group_col]): dict(n=r["n"], selection_rate=r["selection_rate"],
                                      tpr=r["tpr"], fpr=r["fpr"], accuracy=r["accuracy"])
              for r in rows}
    srs = [v["selection_rate"] for v in groups.values() if v["selection_rate"] is not None]
    tprs = [v["tpr"] for v in groups.values() if v["tpr"] is not None]
    fprs = [v["fpr"] for v in groups.values() if v["fpr"] is not None]
    summary = dict(
        demographic_parity_diff=max(srs) - min(srs) if srs else None,
        disparate_impact=min(srs) / max(srs) if srs and max(srs) else None,
        equal_opportunity_diff=max(tprs) - min(tprs) if tprs else None,
        equalized_odds_fpr_diff=max(fprs) - min(fprs) if fprs else None)
    return dict(groups=groups, summary=summary)
print("helpers defined")

helpers defined


## 3. Post-processing thresholding methods

Given scored predictions with per-example score `p1` and a **target selection rate**
$\tau$, we choose, for each subgroup $g$, the threshold $t_g$ equal to the
$(1-\tau)$ quantile of $p1$ within $g$, so that $P(p1 \ge t_g \mid g)\approx\tau$ for every
group. This equalizes selection rate → drives Demographic Parity Difference toward 0.

- **Single-attribute** thresholding conditions on one attribute (Gender *or* Age).
- **Intersectional** thresholding conditions on the **Gender×Age cell**, so *both* marginal
  gaps are corrected at once. This is the method we contribute and evaluate below.

In [4]:
def threshold_pred(pred, group_cols, target):
    # Per-subgroup thresholding to equalize selection rate at `target`.
    # group_cols: columns defining the subgroup (1 col = marginal, 2 = intersectional).
    # Returns (adjusted_pred, {subgroup_key: threshold}).
    key = F.concat_ws("|", *[F.col(c).cast("string") for c in group_cols])
    p = pred.withColumn("__key", key)
    keys = [r["__key"] for r in p.select("__key").distinct().collect()]
    thresholds = {}
    for k in keys:
        sub = p.filter(F.col("__key") == k)
        q = sub.approxQuantile("p1", [1 - target], 0.01)
        thresholds[k] = q[0] if q else 0.5
    mapping = F.create_map(*sum(([F.lit(k), F.lit(v)] for k, v in thresholds.items()), []))
    adj = (p.withColumn("__t", mapping[F.col("__key")])
             .withColumn("pred", (F.col("p1") >= F.col("__t")).cast("double")))
    return adj, thresholds
print("threshold_pred defined")

threshold_pred defined


## 4. Single-seed study (seed 42) — reproduces the D3 story with fairness added

This mirrors the original fairness audit: a leaky model (to quantify leakage inflation),
the honest baseline, and the mitigations — now including intersectional thresholding.

In [5]:
SEED = 42
results = {"meta": {"seed": SEED, "n_total": n_total}}

train, test = df.randomSplit([0.8, 0.2], seed=SEED)
train_fx, test_fx, global_mean = add_train_derived_features(train, test)
results["meta"]["global_train_mean"] = global_mean

# 4a. Leaky reference (averages over full data) vs honest baseline
leaky = add_leaky_features(df)
leaky_train = leaky.join(train.select("UserID", "MovieID"), ["UserID", "MovieID"])
leaky_test = leaky.join(test.select("UserID", "MovieID"), ["UserID", "MovieID"])
leaky_pred = score(train_lr(leaky_train, BASE_FEATURES), leaky_test)
results["leaky_model"] = overall_metrics(leaky_pred)

base_model = train_lr(train_fx, BASE_FEATURES)
base_pred = score(base_model, test_fx).cache()
results["honest_baseline"] = overall_metrics(base_pred)
results["honest_baseline"]["fairness_gender"] = group_fairness(base_pred, "Gender")
results["honest_baseline"]["fairness_age"] = group_fairness(base_pred, "Age")
target_rate = base_pred.select(F.avg("pred")).first()[0]
results["meta"]["target_rate"] = target_rate

print(f"Leaky accuracy : {results['leaky_model']['accuracy']:.4f}")
print(f"Honest accuracy: {results['honest_baseline']['accuracy']:.4f}")
print(f"Gender DPD/DI  : {results['honest_baseline']['fairness_gender']['summary']['demographic_parity_diff']:.3f}"
      f" / {results['honest_baseline']['fairness_gender']['summary']['disparate_impact']:.3f}")
print(f"Age    DPD/DI  : {results['honest_baseline']['fairness_age']['summary']['demographic_parity_diff']:.3f}"
      f" / {results['honest_baseline']['fairness_age']['summary']['disparate_impact']:.3f}")

Leaky accuracy : 0.7234
Honest accuracy: 0.7190
Gender DPD/DI  : 0.036 / 0.946
Age    DPD/DI  : 0.146 / 0.801


In [6]:
# 4b. Mitigations (all evaluated on the seed-42 test split)
# (i) sensitive-feature removal
FAIR_FEATURES = [c for c in BASE_FEATURES if c not in ("gender_encoded", "Age")]
fr_pred = score(train_lr(train_fx, FAIR_FEATURES), test_fx).cache()
results["mitigation_feature_removal"] = overall_metrics(fr_pred)
results["mitigation_feature_removal"]["fairness_gender"] = group_fairness(fr_pred, "Gender")
results["mitigation_feature_removal"]["fairness_age"] = group_fairness(fr_pred, "Age")

# (ii) single-attribute thresholding: Gender-only, then Age-only
g_pred, g_th = threshold_pred(base_pred, ["Gender"], target_rate)
g_pred = g_pred.cache()
results["mitigation_gender_threshold"] = overall_metrics(g_pred)
results["mitigation_gender_threshold"]["fairness_gender"] = group_fairness(g_pred, "Gender")
results["mitigation_gender_threshold"]["fairness_age"] = group_fairness(g_pred, "Age")

a_pred, a_th = threshold_pred(base_pred, ["Age"], target_rate)
a_pred = a_pred.cache()
results["mitigation_age_threshold"] = overall_metrics(a_pred)
results["mitigation_age_threshold"]["fairness_gender"] = group_fairness(a_pred, "Gender")
results["mitigation_age_threshold"]["fairness_age"] = group_fairness(a_pred, "Age")

# (iii) intersectional Gender x Age thresholding (the new method)
x_pred, x_th = threshold_pred(base_pred, ["Gender", "Age"], target_rate)
x_pred = x_pred.cache()
results["mitigation_intersectional_threshold"] = overall_metrics(x_pred)
results["mitigation_intersectional_threshold"]["fairness_gender"] = group_fairness(x_pred, "Gender")
results["mitigation_intersectional_threshold"]["fairness_age"] = group_fairness(x_pred, "Age")
results["mitigation_intersectional_threshold"]["n_cells"] = len(x_th)

import pandas as pd
def row(name, key):
    r = results[key]
    return dict(config=name, accuracy=r["accuracy"], f1=r["f1"],
                gender_DPD=r["fairness_gender"]["summary"]["demographic_parity_diff"],
                gender_DI=r["fairness_gender"]["summary"]["disparate_impact"],
                age_DPD=r["fairness_age"]["summary"]["demographic_parity_diff"],
                age_DI=r["fairness_age"]["summary"]["disparate_impact"])
tbl = pd.DataFrame([
    row("Honest baseline", "honest_baseline"),
    row("Feature removal", "mitigation_feature_removal"),
    row("Gender threshold", "mitigation_gender_threshold"),
    row("Age threshold", "mitigation_age_threshold"),
    row("Intersectional threshold", "mitigation_intersectional_threshold"),
]).set_index("config")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
tbl

,accuracy,f1,gender_DPD,gender_DI,age_DPD,age_DI
config,,,,,,
Honest baseline,0.7190,0.7698,0.0361,0.9464,0.1459,0.8011
Feature removal,0.7187,0.7697,0.0419,0.9382,0.2146,0.7250
Gender threshold,0.7181,0.7699,0.0007,0.9989,0.1457,0.8025
Age threshold,0.7179,0.7697,0.0365,0.9461,0.0043,0.9934
Intersectional threshold,0.7168,0.7687,0.0008,0.9988,0.0031,0.9952


**Read the table above:** single-attribute thresholds fix *their own* attribute but
leave the other's gap roughly at baseline (Gender-only barely moves Age DPD, and vice
versa). **Intersectional thresholding drives both Gender and Age DPD down together**, at a
modest accuracy cost — this is the core method result. The single seed is suggestive; the
next section tests whether it holds across random splits.

## 5. Cross-seed significance testing

A single split can flatter or defame any method. We repeat the full pipeline over **8
independent random 80/20 splits** and report mean±std for each metric, then run **paired**
significance tests (Student's paired $t$ and the non-parametric Wilcoxon signed-rank) on
the seed-matched differences. We test four hypotheses:

- **H1** Intersectional thresholding reduces **Gender DPD** vs. baseline.
- **H2** Intersectional thresholding reduces **Age DPD** vs. baseline.
- **H3** Sensitive-feature removal **increases Age DPD** vs. baseline (unawareness backfires).
- **H4** Gender-only thresholding leaves **Age DPD** essentially unchanged (targeted fix
  does not transfer).

Leakage inflation (leaky vs. honest accuracy) is also tracked across seeds.

In [7]:
SEEDS = [42, 1, 7, 13, 21, 99, 123, 2024]

def run_seed(seed, source_df=None):
    src = df if source_df is None else source_df
    tr, te = src.randomSplit([0.8, 0.2], seed=seed)
    trf, tef, gm = add_train_derived_features(tr, te)
    # leaky reference
    lk = add_leaky_features(src)
    lk_tr = lk.join(tr.select("UserID", "MovieID"), ["UserID", "MovieID"])
    lk_te = lk.join(te.select("UserID", "MovieID"), ["UserID", "MovieID"])
    leaky_acc = overall_metrics(score(train_lr(lk_tr, BASE_FEATURES), lk_te))["accuracy"]
    # honest baseline
    bm = train_lr(trf, BASE_FEATURES)
    bp = score(bm, tef).cache()
    base = overall_metrics(bp)
    bg = group_fairness(bp, "Gender")["summary"]
    ba = group_fairness(bp, "Age")["summary"]
    tgt = bp.select(F.avg("pred")).first()[0]
    # feature removal
    frp = score(train_lr(trf, FAIR_FEATURES), tef)
    fr_age = group_fairness(frp, "Age")["summary"]
    fr_gender = group_fairness(frp, "Gender")["summary"]
    fr_acc = overall_metrics(frp)["accuracy"]
    # gender-only threshold
    gp, _ = threshold_pred(bp, ["Gender"], tgt); gp = gp.cache()
    g_gender = group_fairness(gp, "Gender")["summary"]
    g_age = group_fairness(gp, "Age")["summary"]
    g_acc = overall_metrics(gp)["accuracy"]
    # intersectional threshold
    xp, xth = threshold_pred(bp, ["Gender", "Age"], tgt); xp = xp.cache()
    x_gender = group_fairness(xp, "Gender")["summary"]
    x_age = group_fairness(xp, "Age")["summary"]
    x_acc = overall_metrics(xp)["accuracy"]
    out = dict(
        seed=seed, leaky_acc=leaky_acc, base_acc=base["accuracy"],
        base_gender_DPD=bg["demographic_parity_diff"], base_gender_DI=bg["disparate_impact"],
        base_age_DPD=ba["demographic_parity_diff"], base_age_DI=ba["disparate_impact"],
        fr_acc=fr_acc, fr_age_DPD=fr_age["demographic_parity_diff"],
        fr_gender_DPD=fr_gender["demographic_parity_diff"],
        gthr_acc=g_acc, gthr_gender_DPD=g_gender["demographic_parity_diff"],
        gthr_age_DPD=g_age["demographic_parity_diff"],
        xthr_acc=x_acc, xthr_gender_DPD=x_gender["demographic_parity_diff"],
        xthr_age_DPD=x_age["demographic_parity_diff"],
    )
    for c in (bp, gp, xp):
        c.unpersist()
    return out

t0 = time.time()
rows = []
for s in SEEDS:
    r = run_seed(s)
    rows.append(r)
    print(f"seed {s:>4}: base_acc={r['base_acc']:.4f}  "
          f"gDPD={r['base_gender_DPD']:.3f}  aDPD={r['base_age_DPD']:.3f}  "
          f"xthr gDPD={r['xthr_gender_DPD']:.3f} aDPD={r['xthr_age_DPD']:.3f}  "
          f"({time.time()-t0:.0f}s)")
cs = pd.DataFrame(rows)
cs

seed   42: base_acc=0.7190  gDPD=0.036  aDPD=0.146  xthr gDPD=0.001 aDPD=0.003  (72s)


seed    1: base_acc=0.7170  gDPD=0.036  aDPD=0.148  xthr gDPD=0.001 aDPD=0.003  (133s)


seed    7: base_acc=0.7178  gDPD=0.037  aDPD=0.141  xthr gDPD=0.001 aDPD=0.002  (187s)


seed   13: base_acc=0.7185  gDPD=0.036  aDPD=0.151  xthr gDPD=0.001 aDPD=0.003  (243s)


seed   21: base_acc=0.7175  gDPD=0.041  aDPD=0.138  xthr gDPD=0.000 aDPD=0.002  (310s)


seed   99: base_acc=0.7174  gDPD=0.040  aDPD=0.145  xthr gDPD=0.001 aDPD=0.002  (380s)


seed  123: base_acc=0.7156  gDPD=0.042  aDPD=0.144  xthr gDPD=0.001 aDPD=0.003  (455s)


seed 2024: base_acc=0.7196  gDPD=0.038  aDPD=0.150  xthr gDPD=0.001 aDPD=0.003  (509s)


,seed,leaky_acc,base_acc,base_gender_DPD,base_gender_DI,base_age_DPD,base_age_DI,fr_acc,fr_age_DPD,fr_gender_DPD,gthr_acc,gthr_gender_DPD,gthr_age_DPD,xthr_acc,xthr_gender_DPD,xthr_age_DPD
0,42,0.7234,0.7190,0.0361,0.9464,0.1459,0.8011,0.7187,0.2146,0.0419,0.7181,0.0035,0.1452,0.7168,0.0009,0.0031
1,1,0.7212,0.7170,0.0364,0.9458,0.1480,0.7989,0.7167,0.2154,0.0432,0.7162,0.0002,0.1457,0.7151,0.0006,0.0031
2,7,0.7221,0.7178,0.0373,0.9444,0.1407,0.8061,0.7172,0.2113,0.0421,0.7171,0.0007,0.1407,0.7157,0.0007,0.0018
3,13,0.7222,0.7185,0.0358,0.9467,0.1512,0.7945,0.7179,0.2149,0.0424,0.7177,0.0023,0.1517,0.7163,0.0007,0.0031
4,21,0.7218,0.7175,0.0406,0.9399,0.1378,0.8098,0.7171,0.2078,0.0467,0.7171,0.0008,0.1402,0.7154,0.0000,0.0020
5,99,0.7218,0.7174,0.0400,0.9406,0.1446,0.8028,0.7172,0.2100,0.0447,0.7173,0.0017,0.1445,0.7156,0.0012,0.0023
6,123,0.7195,0.7156,0.0421,0.9374,0.1435,0.8034,0.7152,0.2152,0.0464,0.7149,0.0009,0.1441,0.7136,0.0011,0.0027
7,2024,0.7237,0.7196,0.0384,0.9429,0.1503,0.7952,0.7194,0.2161,0.0443,0.7189,0.0021,0.1503,0.7169,0.0008,0.0026


In [8]:
# Mean +/- std across seeds for the headline quantities
import numpy as np
def ms(col): return f"{cs[col].mean():.4f} +/- {cs[col].std(ddof=1):.4f}"
summary_rows = [
    ("Leaky accuracy",                 ms("leaky_acc")),
    ("Honest baseline accuracy",       ms("base_acc")),
    ("Baseline Gender DPD",            ms("base_gender_DPD")),
    ("Baseline Gender DI",             ms("base_gender_DI")),
    ("Baseline Age DPD",              ms("base_age_DPD")),
    ("Baseline Age DI",               ms("base_age_DI")),
    ("Feature-removal Age DPD",       ms("fr_age_DPD")),
    ("Gender-threshold Gender DPD",   ms("gthr_gender_DPD")),
    ("Gender-threshold Age DPD",      ms("gthr_age_DPD")),
    ("Intersectional Gender DPD",     ms("xthr_gender_DPD")),
    ("Intersectional Age DPD",        ms("xthr_age_DPD")),
    ("Intersectional accuracy",       ms("xthr_acc")),
]
summ = pd.DataFrame(summary_rows, columns=["quantity", "mean +/- std (n=8 seeds)"]).set_index("quantity")
summ

,mean +/- std (n=8 seeds)
quantity,
Leaky accuracy,0.7220 +/- 0.0013
Honest baseline accuracy,0.7178 +/- 0.0012
Baseline Gender DPD,0.0383 +/- 0.0024
Baseline Gender DI,0.9430 +/- 0.0034
Baseline Age DPD,0.1453 +/- 0.0046
Baseline Age DI,0.8015 +/- 0.0052
Feature-removal Age DPD,0.2132 +/- 0.0030
Gender-threshold Gender DPD,0.0015 +/- 0.0011
Gender-threshold Age DPD,0.1453 +/- 0.0041


In [9]:
# Paired significance tests on seed-matched differences
from scipy import stats

def paired(a_col, b_col, alt, label):
    a, b = cs[a_col].values, cs[b_col].values
    d = a - b
    t_stat, t_p = stats.ttest_rel(a, b, alternative=alt)
    try:
        w_stat, w_p = stats.wilcoxon(a, b, alternative=alt)
    except ValueError:
        w_stat, w_p = float("nan"), float("nan")
    return dict(hypothesis=label, mean_diff=d.mean(), t_stat=t_stat,
                t_p=t_p, wilcoxon_p=w_p)

tests = pd.DataFrame([
    paired("base_gender_DPD", "xthr_gender_DPD", "greater",
           "H1: Intersectional < Baseline (Gender DPD)"),
    paired("base_age_DPD", "xthr_age_DPD", "greater",
           "H2: Intersectional < Baseline (Age DPD)"),
    paired("fr_age_DPD", "base_age_DPD", "greater",
           "H3: Feature removal > Baseline (Age DPD, backfires)"),
    paired("gthr_age_DPD", "base_age_DPD", "two-sided",
           "H4: Gender-threshold vs Baseline (Age DPD, ~unchanged)"),
]).set_index("hypothesis")
tests["significant@0.05"] = tests["t_p"] < 0.05
tests

,mean_diff,t_stat,t_p,wilcoxon_p,significant@0.05
hypothesis,,,,,
H1: Intersectional < Baseline (Gender DPD),0.0376,45.1441,0.0000,0.0039,True
H2: Intersectional < Baseline (Age DPD),0.1427,96.1999,0.0000,0.0039,True
"H3: Feature removal > Baseline (Age DPD, backfires)",0.0679,68.7690,0.0000,0.0039,True
"H4: Gender-threshold vs Baseline (Age DPD, ~unchanged)",0.0000,0.0738,0.9433,0.9453,False


The `mean_diff` column is the average seed-matched gap (config A − config B in the
paired call). One-sided tests are used for the directional hypotheses H1–H3; H4 is
two-sided because we expect *no* effect. `significant@0.05` flags where the paired
$t$-test rejects at $\alpha=0.05$; the Wilcoxon column is the non-parametric check.

## 6. Generalization to a second dataset (MovieLens 100K)

To test whether the findings are specific to MovieLens 1M, we replicate the entire
study on **MovieLens 100K** --- an *independent* collection gathered in 1998 (943
different users, 100{,}000 ratings) that, unlike later MovieLens releases, still ships
per-user **gender and age**, so it supports the full pipeline including the
intersectional method. We map its raw integer ages into the same seven MovieLens age
bands and index the string occupation, so every feature and metric matches the 1M setup,
then run the identical 8-seed protocol via the same `run_seed` function.

In [10]:
# ---- Load MovieLens 100K (independent 1998 collection with gender + age) ----
occ_list = sorted({l.split("|")[0].strip() if "|" in l else l.strip()
                   for l in open("ml-100k/u.occupation") if l.strip()})
occ_map = {o: i for i, o in enumerate(occ_list)}
occ_expr = F.create_map(*sum(([F.lit(k), F.lit(v)] for k, v in occ_map.items()), []))

u_schema = StructType([StructField("UserID", IntegerType()), StructField("AgeRaw", IntegerType()),
                       StructField("Gender", StringType()), StructField("Occ", StringType()),
                       StructField("Zip", StringType())])
users100 = spark.read.option("sep", "|").schema(u_schema).csv("ml-100k/u.user")

r_schema = StructType([StructField("UserID", IntegerType()), StructField("MovieID", IntegerType()),
                       StructField("Rating", IntegerType()), StructField("Timestamp", IntegerType())])
ratings100 = spark.read.option("sep", "\t").schema(r_schema).csv("ml-100k/u.data")

item_fields = ([StructField("MovieID", IntegerType()), StructField("Title", StringType()),
                StructField("reldate", StringType()), StructField("videodate", StringType()),
                StructField("url", StringType())]
               + [StructField(f"g{i}", IntegerType()) for i in range(19)])
items100 = spark.read.option("sep", "|").schema(StructType(item_fields)).csv("ml-100k/u.item")
yr_title = F.regexp_extract("Title", r"\((\d{4})\)", 1)
yr_reld = F.regexp_extract("reldate", r"(\d{4})", 1)
items100 = (items100
    .withColumn("num_genres", sum(F.col(f"g{i}") for i in range(19)))
    .withColumn("release_year", F.coalesce(
        F.when(yr_title != "", yr_title.cast(IntegerType())),
        F.when(yr_reld != "", yr_reld.cast(IntegerType())), F.lit(1995)))
    .withColumn("movie_age", F.lit(2000) - F.col("release_year"))
    .select("MovieID", "num_genres", "release_year", "movie_age"))

users100 = (users100
    .withColumn("gender_encoded", (F.col("Gender") == "F").cast(IntegerType()))
    .withColumn("Occupation", occ_expr[F.col("Occ")])
    .withColumn("Age", F.when(F.col("AgeRaw") < 18, 1).when(F.col("AgeRaw") < 25, 18)
                        .when(F.col("AgeRaw") < 35, 25).when(F.col("AgeRaw") < 45, 35)
                        .when(F.col("AgeRaw") < 50, 45).when(F.col("AgeRaw") < 56, 50)
                        .otherwise(56))
    .select("UserID", "Age", "Gender", "gender_encoded", "Occupation"))

r100 = (ratings100.dropDuplicates(["UserID", "MovieID"])
        .filter((F.col("Rating") >= 1) & (F.col("Rating") <= 5))
        .join(users100.select("UserID"), "UserID", "inner")
        .join(items100.select("MovieID"), "MovieID", "inner"))
df100k = (r100.join(users100, "UserID").join(items100, "MovieID")
          .withColumn("high_rating", (F.col("Rating") >= 4).cast(IntegerType())).cache())
n100 = df100k.count()
print(f"ML-100K ratings after cleaning: {n100:,}")
print(f"positive prevalence: {df100k.select(F.avg('high_rating')).first()[0]:.3f}")
print("age bands present:", sorted(r["Age"] for r in df100k.select("Age").distinct().collect()))

ML-100K ratings after cleaning: 100,000
positive prevalence: 0.554


age bands present: [1, 18, 25, 35, 45, 50, 56]


In [11]:
# Same 8-seed protocol, same run_seed function, on the 100K dataframe
t0 = time.time(); rows100 = []
for s in SEEDS:
    r = run_seed(s, df100k); rows100.append(r)
    print(f"[100K] seed {s:>4}: base_acc={r['base_acc']:.4f}  "
          f"gDPD={r['base_gender_DPD']:.3f}  aDPD={r['base_age_DPD']:.3f}  "
          f"xthr gDPD={r['xthr_gender_DPD']:.3f} aDPD={r['xthr_age_DPD']:.3f}  "
          f"({time.time()-t0:.0f}s)")
cs100 = pd.DataFrame(rows100)
cs100

[100K] seed   42: base_acc=0.7047  gDPD=0.003  aDPD=0.180  xthr gDPD=0.002 aDPD=0.008  (20s)


[100K] seed    1: base_acc=0.6978  gDPD=0.007  aDPD=0.086  xthr gDPD=0.000 aDPD=0.003  (38s)


[100K] seed    7: base_acc=0.6986  gDPD=0.002  aDPD=0.143  xthr gDPD=0.000 aDPD=0.005  (56s)


[100K] seed   13: base_acc=0.7023  gDPD=0.015  aDPD=0.186  xthr gDPD=0.000 aDPD=0.003  (76s)


[100K] seed   21: base_acc=0.7009  gDPD=0.004  aDPD=0.179  xthr gDPD=0.001 aDPD=0.004  (97s)


[100K] seed   99: base_acc=0.7038  gDPD=0.011  aDPD=0.197  xthr gDPD=0.002 aDPD=0.005  (117s)


[100K] seed  123: base_acc=0.7031  gDPD=0.006  aDPD=0.159  xthr gDPD=0.002 aDPD=0.005  (135s)


[100K] seed 2024: base_acc=0.6959  gDPD=0.004  aDPD=0.155  xthr gDPD=0.003 aDPD=0.005  (155s)


,seed,leaky_acc,base_acc,base_gender_DPD,base_gender_DI,base_age_DPD,base_age_DI,fr_acc,fr_age_DPD,fr_gender_DPD,gthr_acc,gthr_gender_DPD,gthr_age_DPD,xthr_acc,xthr_gender_DPD,xthr_age_DPD
0,42,0.7183,0.7047,0.0025,0.9958,0.1804,0.7332,0.7062,0.3074,0.0089,0.7050,0.0018,0.1876,0.7020,0.0018,0.0077
1,1,0.7076,0.6978,0.0066,0.9893,0.0864,0.8695,0.6972,0.2170,0.0117,0.6965,0.0013,0.0852,0.6923,0.0005,0.0029
2,7,0.7107,0.6986,0.0016,0.9973,0.1427,0.7825,0.6980,0.2599,0.0105,0.6981,0.0021,0.1408,0.6976,0.0002,0.0048
3,13,0.7148,0.7023,0.0150,0.9762,0.1860,0.7260,0.7030,0.2737,0.0014,0.7021,0.0008,0.1827,0.7005,0.0001,0.0031
4,21,0.7139,0.7009,0.0039,0.9937,0.1787,0.7323,0.7009,0.2880,0.0086,0.7007,0.0037,0.1782,0.7000,0.0010,0.0041
5,99,0.7149,0.7038,0.0107,0.9829,0.1969,0.7107,0.7021,0.3054,0.0015,0.7031,0.0018,0.1991,0.7014,0.0018,0.0052
6,123,0.7148,0.7031,0.0056,0.9909,0.1587,0.7624,0.7038,0.2757,0.0073,0.7031,0.0001,0.1596,0.7010,0.0016,0.0048
7,2024,0.7090,0.6959,0.0039,0.9937,0.1545,0.7723,0.6965,0.2745,0.0061,0.6959,0.0004,0.1527,0.6961,0.0028,0.0052


In [12]:
# Side-by-side 1M vs 100K (mean +/- std over 8 seeds) and 100K significance tests
def ms2(frame, col): return f"{frame[col].mean():.4f} +/- {frame[col].std(ddof=1):.4f}"
comp_spec = [("Baseline accuracy","base_acc"), ("Baseline Gender DPD","base_gender_DPD"),
             ("Baseline Gender DI","base_gender_DI"), ("Baseline Age DPD","base_age_DPD"),
             ("Baseline Age DI","base_age_DI"), ("Feature-removal Age DPD","fr_age_DPD"),
             ("Gender-threshold Age DPD","gthr_age_DPD"),
             ("Intersectional Gender DPD","xthr_gender_DPD"),
             ("Intersectional Age DPD","xthr_age_DPD"), ("Intersectional accuracy","xthr_acc")]
compare = pd.DataFrame([(lbl, ms2(cs, c), ms2(cs100, c)) for lbl, c in comp_spec],
                       columns=["quantity", "MovieLens 1M", "MovieLens 100K"]).set_index("quantity")

def paired2(frame, a_col, b_col, alt, label):
    a, b = frame[a_col].values, frame[b_col].values
    _, t_p = stats.ttest_rel(a, b, alternative=alt)
    try: _, w_p = stats.wilcoxon(a, b, alternative=alt)
    except ValueError: w_p = float("nan")
    return dict(hypothesis=label, mean_diff=(a - b).mean(), t_p=t_p, wilcoxon_p=w_p)
tests100 = pd.DataFrame([
    paired2(cs100, "base_gender_DPD", "xthr_gender_DPD", "greater", "H1: Intersectional < Baseline (Gender)"),
    paired2(cs100, "base_age_DPD", "xthr_age_DPD", "greater", "H2: Intersectional < Baseline (Age)"),
    paired2(cs100, "fr_age_DPD", "base_age_DPD", "greater", "H3: Feature removal > Baseline (Age)"),
    paired2(cs100, "gthr_age_DPD", "base_age_DPD", "two-sided", "H4: Gender-threshold vs Baseline (Age)"),
]).set_index("hypothesis")
print("=== 1M vs 100K ==="); print(compare.to_string())
print("\\n=== 100K significance ==="); print(tests100.to_string())
compare

=== 1M vs 100K ===
                                MovieLens 1M     MovieLens 100K
quantity                                                       
Baseline accuracy          0.7178 +/- 0.0012  0.7009 +/- 0.0032
Baseline Gender DPD        0.0383 +/- 0.0024  0.0062 +/- 0.0045
Baseline Gender DI         0.9430 +/- 0.0034  0.9900 +/- 0.0071
Baseline Age DPD           0.1453 +/- 0.0046  0.1605 +/- 0.0350
Baseline Age DI            0.8015 +/- 0.0052  0.7611 +/- 0.0503
Feature-removal Age DPD    0.2132 +/- 0.0030  0.2752 +/- 0.0286
Gender-threshold Age DPD   0.1453 +/- 0.0041  0.1607 +/- 0.0362
Intersectional Gender DPD  0.0008 +/- 0.0004  0.0012 +/- 0.0010
Intersectional Age DPD     0.0026 +/- 0.0005  0.0047 +/- 0.0015
Intersectional accuracy    0.7157 +/- 0.0011  0.6989 +/- 0.0033
\n=== 100K significance ===
                                        mean_diff    t_p  wilcoxon_p
hypothesis                                                          
H1: Intersectional < Baseline (Gender)     0.00

,MovieLens 1M,MovieLens 100K
quantity,,
Baseline accuracy,0.7178 +/- 0.0012,0.7009 +/- 0.0032
Baseline Gender DPD,0.0383 +/- 0.0024,0.0062 +/- 0.0045
Baseline Gender DI,0.9430 +/- 0.0034,0.9900 +/- 0.0071
Baseline Age DPD,0.1453 +/- 0.0046,0.1605 +/- 0.0350
Baseline Age DI,0.8015 +/- 0.0052,0.7611 +/- 0.0503
Feature-removal Age DPD,0.2132 +/- 0.0030,0.2752 +/- 0.0286
Gender-threshold Age DPD,0.1453 +/- 0.0041,0.1607 +/- 0.0362
Intersectional Gender DPD,0.0008 +/- 0.0004,0.0012 +/- 0.0010
Intersectional Age DPD,0.0026 +/- 0.0005,0.0047 +/- 0.0015


## 7. Figures for the paper

In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("paper", exist_ok=True)
cfgs = ["Honest\nbaseline", "Feature\nremoval", "Gender\nthreshold",
        "Age\nthreshold", "Intersect.\nthreshold"]
keys = ["honest_baseline", "mitigation_feature_removal", "mitigation_gender_threshold",
        "mitigation_age_threshold", "mitigation_intersectional_threshold"]
g_dpd = [results[k]["fairness_gender"]["summary"]["demographic_parity_diff"] for k in keys]
a_dpd = [results[k]["fairness_age"]["summary"]["demographic_parity_diff"] for k in keys]
accs = [results[k]["accuracy"] for k in keys]

# Fig 1: Gender vs Age DPD across the 5 configurations (seed 42)
x = np.arange(len(cfgs)); w = 0.38
fig, ax = plt.subplots(figsize=(6.4, 3.6))
b1 = ax.bar(x - w/2, g_dpd, w, label="Gender DPD", color="#4C72B0")
b2 = ax.bar(x + w/2, a_dpd, w, label="Age DPD", color="#DD8452")
ax.set_ylabel("Demographic Parity Difference\n(lower is fairer)")
ax.set_xticks(x); ax.set_xticklabels(cfgs, fontsize=8)
ax.set_title("Both gaps fall only under intersectional thresholding")
ax.legend(fontsize=8)
for b in list(b1) + list(b2):
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()),
                ha="center", va="bottom", fontsize=6.5)
plt.tight_layout(); plt.savefig("paper/fig_multi_dpd.png", dpi=220); plt.show()

/var/folders/b0/ytc83zrj7sqdn_81z1zvv26r0000gn/T/ipykernel_44306/457594246.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("paper/fig_multi_dpd.png", dpi=220); plt.show()


In [14]:
# Fig 2: cross-seed distribution of Age DPD (baseline / feature-removal / gender-thr / intersectional)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
box_data = [cs["base_age_DPD"], cs["fr_age_DPD"], cs["gthr_age_DPD"], cs["xthr_age_DPD"]]
box_labels = ["Baseline", "Feature\nremoval", "Gender\nthreshold", "Intersect.\nthreshold"]
bp2 = ax.boxplot(box_data, showmeans=True, patch_artist=True)
ax.set_xticks(range(1, len(box_labels) + 1))
ax.set_xticklabels(box_labels)
for patch, c in zip(bp2["boxes"], ["#BBBBBB", "#DD8452", "#4C72B0", "#55A868"]):
    patch.set_facecolor(c)
ax.set_ylabel("Age Demographic Parity Difference")
ax.set_title("Age DPD across 8 seeds: unawareness worsens it, intersectional fixes it")
plt.tight_layout(); plt.savefig("paper/fig_crossseed_age.png", dpi=220); plt.show()

/var/folders/b0/ytc83zrj7sqdn_81z1zvv26r0000gn/T/ipykernel_44306/3276847026.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("paper/fig_crossseed_age.png", dpi=220); plt.show()


In [15]:
# Fig 3: fairness-accuracy tradeoff across seeds (intersectional), mean +/- std
fig, ax = plt.subplots(figsize=(6.4, 3.6))
pts = [("Baseline", cs["base_acc"], cs["base_gender_DPD"], "#BBBBBB"),
       ("Gender thr.", cs["gthr_acc"], cs["gthr_gender_DPD"], "#4C72B0"),
       ("Intersect. thr.", cs["xthr_acc"], cs["xthr_gender_DPD"], "#55A868")]
for name, ac, dpd, c in pts:
    ax.errorbar(dpd.mean(), ac.mean(), xerr=dpd.std(ddof=1), yerr=ac.std(ddof=1),
                fmt="o", ms=9, color=c, capsize=3, label=name)
    ax.annotate(name, (dpd.mean(), ac.mean()), textcoords="offset points",
                xytext=(8, 4), fontsize=8)
ax.set_xlabel("Gender Demographic Parity Difference (lower is fairer)")
ax.set_ylabel("Overall accuracy")
ax.set_title("Fairness-accuracy tradeoff (mean +/- std over 8 seeds)")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("paper/fig_tradeoff_seeds.png", dpi=220); plt.show()

/var/folders/b0/ytc83zrj7sqdn_81z1zvv26r0000gn/T/ipykernel_44306/3242702153.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("paper/fig_tradeoff_seeds.png", dpi=220); plt.show()


In [16]:
# Fig 4: cross-dataset replication -- baseline vs intersectional Age DPD, 1M vs 100K
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ds = ["MovieLens 1M", "MovieLens 100K"]
base_age = [cs["base_age_DPD"].mean(), cs100["base_age_DPD"].mean()]
base_err = [cs["base_age_DPD"].std(ddof=1), cs100["base_age_DPD"].std(ddof=1)]
xthr_age = [cs["xthr_age_DPD"].mean(), cs100["xthr_age_DPD"].mean()]
xthr_err = [cs["xthr_age_DPD"].std(ddof=1), cs100["xthr_age_DPD"].std(ddof=1)]
x = np.arange(2); w = 0.35
ax.bar(x - w/2, base_age, w, yerr=base_err, capsize=3, label="Baseline", color="#DD8452")
ax.bar(x + w/2, xthr_age, w, yerr=xthr_err, capsize=3, label="Intersectional threshold",
       color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(ds)
ax.set_ylabel("Age Demographic Parity Difference")
ax.set_title("Age bias and its intersectional fix replicate across datasets")
ax.legend(fontsize=8)
for i, (b, xt) in enumerate(zip(base_age, xthr_age)):
    ax.annotate(f"{b:.3f}", (i - w/2, b), ha="center", va="bottom", fontsize=7)
    ax.annotate(f"{xt:.3f}", (i + w/2, xt), ha="center", va="bottom", fontsize=7)
plt.tight_layout(); plt.savefig("paper/fig_dataset_compare.png", dpi=220); plt.show()

/var/folders/b0/ytc83zrj7sqdn_81z1zvv26r0000gn/T/ipykernel_44306/667175269.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("paper/fig_dataset_compare.png", dpi=220); plt.show()


## 8. Persist results

In [17]:
results["cross_seed"] = {
    "seeds": SEEDS,
    "per_seed": rows,
    "summary": {k: {"mean": float(cs[k].mean()), "std": float(cs[k].std(ddof=1))}
                for k in cs.columns if k != "seed"},
    "significance": tests.reset_index().to_dict(orient="records"),
}
results["ml100k"] = {
    "n_total": n100,
    "seeds": SEEDS,
    "per_seed": rows100,
    "summary": {k: {"mean": float(cs100[k].mean()), "std": float(cs100[k].std(ddof=1))}
                for k in cs100.columns if k != "seed"},
    "significance": tests100.reset_index().to_dict(orient="records"),
}
with open("fairness_results_d4.json", "w") as f:
    json.dump(results, f, indent=2, default=float)
cs.to_csv("fairness_cross_seed.csv", index=False)
cs100.to_csv("fairness_cross_seed_ml100k.csv", index=False)
print("wrote fairness_results_d4.json, fairness_cross_seed.csv, fairness_cross_seed_ml100k.csv,")
print("      paper/fig_multi_dpd.png, fig_crossseed_age.png, fig_tradeoff_seeds.png, fig_dataset_compare.png")

wrote fairness_results_d4.json, fairness_cross_seed.csv, fairness_cross_seed_ml100k.csv,
      paper/fig_multi_dpd.png, fig_crossseed_age.png, fig_tradeoff_seeds.png, fig_dataset_compare.png


## 9. Findings (all from real runs)

- **Leakage is modest but consistent** — the leaky averages inflate accuracy by a small,
  stable margin across seeds (see §5 summary), confirming the D3 numbers were only mildly
  optimistic.
- **Gender bias is mild; age bias is substantial** — the baseline Age DPD/DI sit far from
  parity while Gender is close, and this holds across all 8 seeds.
- **Fairness through unawareness backfires for age (H3)** — dropping the protected
  attributes *raises* Age DPD, because proxies (occupation, movie-age preferences) remain.
- **Single-attribute thresholding does not transfer (H4)** — fixing Gender leaves Age DPD
  essentially unchanged.
- **Intersectional thresholding fixes both gaps (H1, H2)** — conditioning on Gender×Age
  cells drives *both* marginal DPDs down together at a small accuracy cost, and the paired
  tests confirm the reductions are significant, not seed noise.
- **The findings replicate on MovieLens 100K (§6)** — an independent 1998 collection shows
  the same substantial baseline age gap and the same intersectional fix, so the results are
  not specific to the 1M dataset.

See the significance tables in §5 and §6 for exact p-values; `fairness_results_d4.json`
holds the full numeric record for both datasets.

In [18]:
spark.stop()